# **ENVIROMENT INITIALIZATION**

In [1]:
# IMPORTS
# Math libraries
import numpy as np
import pandas as pd
import polars as pl

# ML libraries
import lightgbm as lgb

# Technical libraries
from pathlib import Path
from datetime import date, timedelta

In [2]:
# CONFIGURATION
from config import (
    PROJECT_ROOT,
    TRAIN_PATH,
    TARGET_AS_OF,
    DATA_PROCESSED_DIR, DATA_FEATURES_DIR, DATA_SUBMISSIONS_DIR,
    TARGET_FEATURES_PATH,
    MODELS_DIR,
    WINDOWS
)

In [3]:
# DATA LOADING
lf_full = pl.scan_parquet(TRAIN_PATH)

# Filter to final AS_OF
lf_final = lf_full.filter(pl.col("event_date") <= TARGET_AS_OF)

print(f"Data filtered to {TARGET_AS_OF}")

Data filtered to 2026-02-13


# **BUILDING FEATURES FOR TARGET**

In [4]:
# RECENCY FEATURES
recency_features_final = (
    lf_final.group_by("user_id")
    .agg([
        pl.col("event_date").max().alias("last_activity_date"),
        pl.col("event_date").filter(pl.col("to_ord") > 0).max().alias("last_order_date"),
        pl.col("event_date").filter(pl.col("to_cart") > 0).max().alias("last_cart_date"),
        pl.col("event_date").filter(pl.col("search") == 1).max().alias("last_search_date"),
        pl.col("event_date").filter(pl.col("cat") == 1).max().alias("last_cat_date"),
    ])
    .with_columns([
        (pl.lit(TARGET_AS_OF) - pl.col("last_activity_date")).dt.total_days().alias("recency_activity"),
        (pl.lit(TARGET_AS_OF) - pl.col("last_order_date")).dt.total_days().alias("recency_order"),
        (pl.lit(TARGET_AS_OF) - pl.col("last_cart_date")).dt.total_days().alias("recency_cart"),
        (pl.lit(TARGET_AS_OF) - pl.col("last_search_date")).dt.total_days().alias("recency_search"),
        (pl.lit(TARGET_AS_OF) - pl.col("last_cat_date")).dt.total_days().alias("recency_cat"),
    ])
    .drop(["last_activity_date", "last_order_date", "last_cart_date", "last_search_date", "last_cat_date"])
    .collect(engine = "streaming")
)
recency_features_final = recency_features_final.fill_null(999)

# FREQUENCY FEATURES
frequency_exprs_final = []
for w in WINDOWS:
    window_start = TARGET_AS_OF - timedelta(days = w)
    frequency_exprs_final.extend([
        pl.col("event_date").filter(
            (pl.col("event_date") >= window_start) & (pl.col("event_date") <= TARGET_AS_OF)
        ).n_unique().alias(f"active_days_{w}d"),
        pl.col("to_ord").filter(
            (pl.col("event_date") >= window_start) & (pl.col("event_date") <= TARGET_AS_OF)
        ).sum().alias(f"orders_{w}d"),
        pl.col("to_cart").filter(
            (pl.col("event_date") >= window_start) & (pl.col("event_date") <= TARGET_AS_OF)
        ).sum().alias(f"cart_adds_{w}d"),
        pl.col("searches").filter(
            (pl.col("event_date") >= window_start) & (pl.col("event_date") <= TARGET_AS_OF)
        ).sum().alias(f"searches_{w}d"),
    ])
frequency_features_final = (
    lf_final.group_by("user_id").agg(frequency_exprs_final).collect()
)

# MONETARY FEATURES
monetary_exprs_final = []
for w in WINDOWS:
    window_start = TARGET_AS_OF - timedelta(days = w)
    monetary_exprs_final.extend([
        pl.col("gmv").filter(
            (pl.col("event_date") >= window_start) & (pl.col("event_date") <= TARGET_AS_OF)
        ).sum().alias(f"gmv_{w}d"),
        pl.col("gmv_search").filter(
            (pl.col("event_date") >= window_start) & (pl.col("event_date") <= TARGET_AS_OF)
        ).sum().alias(f"gmv_search_{w}d"),
        pl.col("gmv_cat").filter(
            (pl.col("event_date") >= window_start) & (pl.col("event_date") <= TARGET_AS_OF)
        ).sum().alias(f"gmv_cat_{w}d"),
    ])
monetary_features_final = (
    lf_final.group_by("user_id").agg(monetary_exprs_final).collect(engine = "streaming")
)

# DAY-OF-WEEK FEATURES
lf_final_dow = lf_final.with_columns(pl.col("event_date").dt.weekday().alias("weekday"))
dow_features_final = (
    lf_final_dow.group_by("user_id")
    .agg([
        pl.col("weekday").filter((pl.col("weekday") == 6) | (pl.col("weekday") == 7)).count().alias("weekend_days"),
        pl.col("weekday").filter((pl.col("weekday") >= 1) & (pl.col("weekday") <= 5)).count().alias("weekday_days"),
        pl.col("to_ord").filter(pl.col("weekday") == 2).sum().alias("tuesday_orders"),
        pl.col("to_ord").filter(pl.col("weekday") == 6).sum().alias("saturday_orders"),
        pl.col("to_ord").filter(pl.col("weekday") == 7).sum().alias("sunday_orders"),
        pl.col("to_ord").filter((pl.col("weekday") >= 1) & (pl.col("weekday") <= 5)).sum().alias("weekday_orders"),
        pl.col("to_ord").filter((pl.col("weekday") >= 6) & (pl.col("weekday") <= 7)).sum().alias("weekend_orders"),
        pl.col("gmv").filter(pl.col("weekday") == 2).sum().alias("tuesday_gmv"),
        pl.col("gmv").filter(pl.col("weekday") == 6).sum().alias("saturday_gmv"),
        pl.col("gmv").filter(pl.col("weekday") == 7).sum().alias("sunday_gmv"),
        pl.col("gmv").filter((pl.col("weekday") >= 1) & (pl.col("weekday") <= 5)).sum().alias("weekday_gmv"),
        pl.col("gmv").filter((pl.col("weekday") >= 6) & (pl.col("weekday") <= 7)).sum().alias("weekend_gmv"),
    ])
    .with_columns([
        (pl.col("weekend_orders") / (pl.col("weekend_orders") + pl.col("weekday_orders"))).fill_null(0).alias("weekend_order_share"),
        (pl.col("weekend_gmv") / (pl.col("weekend_gmv") + pl.col("weekday_gmv"))).fill_null(0).alias("weekend_gmv_share"),
    ])
    .drop(["total_orders_dow", "total_gmv_dow"], strict = False)
    .collect(engine = "streaming")
)

# CONVERSION FEATURES
conversion_features_final = (
    lf_final.group_by("user_id")
    .agg([
        pl.col("search_to_cart").sum().alias("search_to_cart_total"),
        pl.col("search_to_ord").sum().alias("search_to_ord_total"),
        pl.col("cat_to_cart").sum().alias("cat_to_cart_total"),
        pl.col("cat_to_ord").sum().alias("cat_to_ord_total"),
        pl.col("searches").sum().alias("searches_total"),
        pl.col("to_cart").sum().alias("cart_adds_total"),
        pl.col("to_ord").sum().alias("orders_total"),
        pl.col("search").sum().alias("search_days_total"),
        pl.col("cat").sum().alias("cat_days_total"),
    ])
    .with_columns([
        (pl.col("search_to_cart_total") / pl.col("searches_total")).fill_null(0).alias("search_to_cart_rate"),
        (pl.col("search_to_ord_total") / pl.col("searches_total")).fill_null(0).alias("search_to_ord_rate"),
        (pl.col("orders_total") / pl.col("cart_adds_total")).fill_null(0).alias("cart_to_ord_rate"),
        (pl.col("cat_to_cart_total") / pl.col("cat_days_total")).fill_null(0).alias("cat_to_cart_rate"),
        (pl.col("cat_to_ord_total") / pl.col("cat_days_total")).fill_null(0).alias("cat_to_ord_rate"),
        (pl.col("orders_total") / pl.col("search_days_total")).fill_null(0).alias("orders_per_active_day"),
        (pl.col("cart_adds_total") / pl.col("search_days_total")).fill_null(0).alias("carts_per_active_day"),
    ])
    .drop([
        "search_to_cart_total", "search_to_ord_total", "cat_to_cart_total", "cat_to_ord_total",
        "searches_total", "cart_adds_total", "orders_total", "search_days_total", "cat_days_total",
    ])
    .collect(engine = "streaming")
)

# SEASONAL FEATURES
lf_final_seasonal = lf_final.with_columns(pl.col("event_date").dt.day().alias("day_of_month"))
seasonal_features_final = (
    lf_final_seasonal.group_by("user_id")
    .agg([
        pl.col("to_ord").filter(pl.col("day_of_month").is_in([10, 11, 25, 26])).sum().alias("salary_day_orders"),
        pl.col("to_ord").filter(~pl.col("day_of_month").is_in([10, 11, 25, 26])).sum().alias("non_salary_day_orders"),
        pl.col("gmv").filter(pl.col("day_of_month").is_in([10, 11, 25, 26])).sum().alias("salary_day_gmv"),
        pl.col("gmv").filter(~pl.col("day_of_month").is_in([10, 11, 25, 26])).sum().alias("non_salary_day_gmv"),
        pl.col("to_ord").sum().alias("total_orders"),
        pl.col("gmv").sum().alias("total_gmv"),
    ])
    .with_columns([
        (pl.col("salary_day_orders") / pl.col("total_orders")).fill_null(0).alias("salary_day_order_share"),
        (pl.col("salary_day_gmv") / pl.col("total_gmv")).fill_null(0).alias("salary_day_gmv_share"),
        (pl.col("salary_day_gmv") / pl.col("salary_day_orders")).fill_null(0).alias("salary_day_aov"),
        (pl.col("non_salary_day_gmv") / pl.col("non_salary_day_orders")).fill_null(0).alias("non_salary_day_aov"),
    ])
    .drop(["total_orders", "total_gmv"])
    .collect(engine = "streaming")
)

# CHANNEL FEATURES
channel_features_final = (
    monetary_features_final.join(frequency_features_final, on = "user_id", how = "inner")
    .with_columns([
        (pl.col("gmv_search_30d") / pl.col("gmv_30d")).fill_null(0).alias("search_gmv_share_30d"),
        (pl.col("gmv_cat_30d") / pl.col("gmv_30d")).fill_null(0).alias("cat_gmv_share_30d"),
        (pl.col("gmv_search_90d") / pl.col("gmv_90d")).fill_null(0).alias("search_gmv_share_90d"),
        (pl.col("gmv_cat_90d") / pl.col("gmv_90d")).fill_null(0).alias("cat_gmv_share_90d"),
        ((pl.col("gmv_search_30d") > 0) & (pl.col("gmv_cat_30d") > 0)).cast(pl.Int32).alias("uses_both_channels_30d"),
        ((pl.col("gmv_search_90d") > 0) & (pl.col("gmv_cat_90d") > 0)).cast(pl.Int32).alias("uses_both_channels_90d"),
    ])
)

# COMBINE ALL FEATURES
all_features_final = recency_features_final
all_features_final = all_features_final.join(frequency_features_final, on = "user_id", how = "inner")
all_features_final = all_features_final.join(monetary_features_final, on = "user_id", how = "inner")
all_features_final = all_features_final.join(dow_features_final, on = "user_id", how = "inner")
all_features_final = all_features_final.join(conversion_features_final, on = "user_id", how = "inner")
all_features_final = all_features_final.join(seasonal_features_final, on = "user_id", how = "inner")
all_features_final = all_features_final.join(channel_features_final, on = "user_id", how = "inner")

# TREND FEATURES
trend_features_final = (
    frequency_features_final.join(monetary_features_final, on = "user_id", how = "inner")
    .with_columns([
        (pl.col("active_days_7d") / (pl.col("active_days_30d") / 4)).fill_null(1).alias("activity_trend_7_30"),
        (pl.col("active_days_14d") / (pl.col("active_days_30d") / 2)).fill_null(1).alias("activity_trend_14_30"),
        (pl.col("active_days_30d") / (pl.col("active_days_90d") / 3)).fill_null(1).alias("activity_trend_30_90"),
        (pl.col("orders_7d") / (pl.col("orders_30d") / 4)).fill_null(1).alias("orders_trend_7_30"),
        (pl.col("orders_14d") / (pl.col("orders_30d") / 2)).fill_null(1).alias("orders_trend_14_30"),
        (pl.col("orders_30d") / (pl.col("orders_90d") / 3)).fill_null(1).alias("orders_trend_30_90"),
        (pl.col("gmv_7d") / (pl.col("gmv_30d") / 4)).fill_null(1).alias("gmv_trend_7_30"),
        (pl.col("gmv_14d") / (pl.col("gmv_30d") / 2)).fill_null(1).alias("gmv_trend_14_30"),
        (pl.col("gmv_30d") / (pl.col("gmv_90d") / 3)).fill_null(1).alias("gmv_trend_30_90"),
    ])
)
trend_cols = ["user_id"] + [c for c in trend_features_final.columns if "trend" in c]
trend_subset_final = trend_features_final.select(trend_cols)
all_features_final = all_features_final.join(trend_subset_final, on = "user_id", how = "inner")

# Fill nulls
all_features_final = all_features_final.fill_null(0)

print(f"Final features shape: {all_features_final.shape}")

# Save final features
DATA_FEATURES_DIR.mkdir(parents = True, exist_ok = True)
all_features_final.write_parquet(TARGET_FEATURES_PATH)
print(f"Final features saved to: {TARGET_FEATURES_PATH}")

Final features shape: (250000, 134)
Final features saved to: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\data\processed\features\target_features_2026-02-13.parquet


# **MAKING PREDICTIONS**

In [8]:
# LOADING MODELS
clf_model_name = "clf_lightgbm_20260824_181511.txt"
reg_model_name = "reg_lightgbm_20260824_181511.txt"

clf_model_path = MODELS_DIR / "01_lgbm_clf_reg" / clf_model_name
reg_model_path = MODELS_DIR / "01_lgbm_clf_reg" / reg_model_name

if not clf_model_path.exists():
    print(f"ERROR: Classifier model not found: {clf_model_path}")
elif not reg_model_path.exists():
    print(f"ERROR: Regressor model not found: {reg_model_path}")
else:
    print(f"Loading classifier: {clf_model_path}")
    print(f"Loading regressor: {reg_model_path}")
    
    clf_final = lgb.Booster(model_file=str(clf_model_path))
    reg_final = lgb.Booster(model_file=str(reg_model_path))
    
    print("Models loaded successfully!")
    
    # MAKING PREDICTIONS    
    # Remove user_id for prediction
    feature_cols_final = [c for c in all_features_final.columns if c != "user_id"]
    X_final = all_features_final[feature_cols_final].to_numpy()
    
    # Classifier predictions
    P_buy_final = clf_final.predict(X_final)
    
    # Regressor predictions
    predicted_gmv_log_final = reg_final.predict(X_final)
    predicted_gmv_final = np.expm1(predicted_gmv_log_final)
    
    # Hurdle model: P(buy) * predicted_gmv
    final_predictions = P_buy_final * predicted_gmv_final
    final_predictions = np.clip(final_predictions, 0, None)
    
    # CREATING SUBMISSION    
    submission_df = pd.DataFrame({
        "user_id": all_features_final["user_id"].to_numpy(),
        "predict": final_predictions,
    })
    
    # Sort by user_id
    submission_df = submission_df.sort_values("user_id")
    
    # Save submission
    submission_path = DATA_SUBMISSIONS_DIR / "submission_01.csv"
    submission_df.to_csv(submission_path, index = False)
    
    print(f"\nSubmission saved to: {submission_path}")
    print(f"Number of predictions: {len(submission_df)}")
    print(f"\nFirst 10 predictions:")
    print(submission_df.head(10))
    
    print(f"\nPrediction statistics:")
    print(f"  Mean: {final_predictions.mean():.4f}")
    print(f"  Median: {np.median(final_predictions):.4f}")
    print(f"  Min: {final_predictions.min():.4f}")
    print(f"  Max: {final_predictions.max():.4f}")
    print(f"  Zeros: {(final_predictions == 0).sum()}")
    print(f"  Positive: {(final_predictions > 0).sum()}")

Loading classifier: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\models\01_lgbm_clf_reg\clf_lightgbm_20260824_181511.txt
Loading regressor: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\models\01_lgbm_clf_reg\reg_lightgbm_20260824_181511.txt
Models loaded successfully!

Submission saved to: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\data\submissions\submission_01.csv
Number of predictions: 250000

First 10 predictions:
        user_id     predict
27837         2    8.578860
90556         7  111.911967
153294       15   19.286179
153151       18  161.728424
27998        23    3.510298
215594       26    3.241463
153110       27   15.794477
153243       30    4.095234
27812        34    4.253651
58893        37   14.828126

Prediction statistics:
  Mean: 49.3402
  Median: 20.2246
  Min: 0.3654
  Max: 3175.4572
  Zeros: 0
  Positive: 250000
